In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
# Load Dataset
dataset = pd.read_csv("CKD.csv")

In [4]:
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [5]:
# Independent and Dependent Variables
Independent = dataset.drop(columns=['classification'])
Dependent = dataset['classification']

In [6]:

# Convert categorical columns
Independent = pd.get_dummies(Independent, drop_first=True)



In [7]:
# Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(Independent, Dependent, test_size=1/3, random_state=0)



In [8]:
# Feature Scaling (Important for KNN)
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)



In [9]:
# Grid Search Parameters for KNN
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': [1, 2]
}
grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    refit=True,
    verbose=3,
    n_jobs=-1,
    scoring='f1_weighted'
)
grid.fit(X_train, y_train)
# Results
re = grid.cv_results_



Fitting 5 folds for each of 84 candidates, totalling 420 fits


/Users/spark/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:824: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/spark/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 813, in _score
    scores = scorer(estimator, X_test, y_test)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/spark/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 266, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/spark/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 353, in _score
    y_pred = method_caller(estimator, "predict", X)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/spark/anaconda3/lib/python

In [10]:
# Prediction
grid_predictions = grid.predict(X_test)



In [11]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



In [12]:
# F1 Score
from sklearn.metrics import f1_score
f1_weighted = f1_score(
    y_test,
    grid_predictions,
    average='weighted'
)



In [13]:
print("The f1_weighted value for best parameter {}:".format(grid.best_params_),f1_weighted)
print("The Confusion Matrix:\n", cm)



The f1_weighted value for best parameter {'metric': 'manhattan', 'n_neighbors': 3, 'p': 1, 'weights': 'distance'}: 0.9701163285572423
The Confusion Matrix:
 [[51  0]
 [ 4 78]]


In [14]:
# ROC-AUC
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(
    y_test,
    grid.predict_proba(X_test)[:, 1]
)
print("ROC_AUC Score:", roc_auc)



ROC_AUC Score: 0.9875657580105213


In [15]:
# Grid Search Results Table
table = pd.DataFrame.from_dict(re)



In [16]:
# Classification Report
from sklearn.metrics import classification_report
clf_report = classification_report(y_test,grid_predictions)
print(clf_report)

              precision    recall  f1-score   support

          no       0.93      1.00      0.96        51
         yes       1.00      0.95      0.97        82

    accuracy                           0.97       133
   macro avg       0.96      0.98      0.97       133
weighted avg       0.97      0.97      0.97       133



In [21]:
print("OVERALL REPORT for KNN Classification:\n")
print("Best Parameters:\n\t", grid.best_params_)
print("Best Cross Validation Score:\n\t", grid.best_score_)
print("The f1_weighted value for best parameter:\n\t",f1_weighted)
print("The confusion Matrix:\n",cm)
print("ROC_AUC Score:\n\t", roc_auc)
print("classification_report:\n\t",clf_report)

OVERALL REPORT for KNN Classification:

Best Parameters:
	 {'metric': 'manhattan', 'n_neighbors': 3, 'p': 1, 'weights': 'distance'}
Best Cross Validation Score:
	 0.9739513796952457
The f1_weighted value for best parameter:
	 0.9701163285572423
The confusion Matrix:
 [[51  0]
 [ 4 78]]
ROC_AUC Score:
	 0.9875657580105213
classification_report:
	               precision    recall  f1-score   support

          no       0.93      1.00      0.96        51
         yes       1.00      0.95      0.97        82

    accuracy                           0.97       133
   macro avg       0.96      0.98      0.97       133
weighted avg       0.97      0.97      0.97       133

